# Age Classification - 5-Fold Cross Validation
## Using Fine-tuned Model (Previous age_classifier.pth)

**Method:** Video-based split (no overlap)
**Model:** Previously fine-tuned EfficientNet-B0

## Step 1: Setup

In [10]:
!pip install torch torchvision timm scikit-learn seaborn -q

from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Using: cuda


## Step 2: Configuration

In [11]:
BASE_PATH = Path("/content/drive/MyDrive/face_pipeline_project")
LABELS_FILE = BASE_PATH / "ground_truth/age_labels_v2.csv"
MODEL_PATH = BASE_PATH / "models/age_classifier.pth"  # Previously fine-tuned model
METRICS_FOLDER = BASE_PATH / "metrics"

N_FOLDS = 5
BATCH_SIZE = 32
IMAGE_SIZE = 224

print(f"Model: {MODEL_PATH}")
print(f"Labels: {LABELS_FILE}")

Model: /content/drive/MyDrive/face_pipeline_project/models/age_classifier.pth
Labels: /content/drive/MyDrive/face_pipeline_project/ground_truth/age_labels_manual.csv


## Step 3: Load Fine-tuned Model

In [12]:
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=2)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

print("✓ Fine-tuned model loaded")

✓ Fine-tuned model loaded


## Step 4: Load Labels & Create Folds

In [13]:
df = pd.read_csv(LABELS_FILE)
df = df.drop_duplicates(subset=['image_path'], keep='last')
df = df[df['label'].isin(['adult', 'child'])].reset_index(drop=True)
df['label_int'] = (df['label'] == 'adult').astype(int)

print(f"Total samples: {len(df)}")
print(f"Videos: {df['video'].nunique()}")

# Create video folds
videos = df['video'].unique()
np.random.seed(42)
np.random.shuffle(videos)

fold_size = len(videos) // N_FOLDS
video_folds = []
for i in range(N_FOLDS):
    if i == N_FOLDS - 1:
        fold_videos = videos[i * fold_size:]
    else:
        fold_videos = videos[i * fold_size:(i + 1) * fold_size]
    video_folds.append(list(fold_videos))
    print(f"Fold {i+1}: {len(fold_videos)} videos")

Total samples: 1283
Videos: 27
Fold 1: 5 videos
Fold 2: 5 videos
Fold 3: 5 videos
Fold 4: 5 videos
Fold 5: 7 videos


## Step 5: Dataset & Transform

In [14]:
class FaceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label = row['label_int']
        if self.transform:
            image = self.transform(image)
        return image, label

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("✓ Dataset ready")

✓ Dataset ready


## Step 6: Run 5-Fold Evaluation (Inference Only)

In [15]:
fold_results = []
all_true = []
all_pred = []

for fold_num in range(1, N_FOLDS + 1):
    test_videos = video_folds[fold_num - 1]
    test_df = df[df['video'].isin(test_videos)].copy()

    print(f"\nFold {fold_num}: Testing on {len(test_df)} samples from {len(test_videos)} videos")

    test_dataset = FaceDataset(test_df, transform)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    fold_preds = []
    fold_labels = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"Fold {fold_num}"):
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            fold_preds.extend(predicted.cpu().numpy())
            fold_labels.extend(labels.numpy())

    acc = accuracy_score(fold_labels, fold_preds)
    fold_results.append({'fold': fold_num, 'samples': len(test_df), 'accuracy': acc})

    all_true.extend(fold_labels)
    all_pred.extend(fold_preds)

    print(f"  Accuracy: {acc:.2%}")

print("\n✓ All folds complete")


Fold 1: Testing on 340 samples from 5 videos


Fold 1:   0%|          | 0/11 [00:00<?, ?it/s]


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipython-input-3293105469.py", line 11, in __getitem__
    image = Image.open(row['image_path']).convert('RGB')
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/face_pipeline_project/face_crops_aligned/10.01.00-11.05.46[M][0@0][1

## Step 7: Results Summary

In [ ]:
results_df = pd.DataFrame(fold_results)
print(results_df.to_string(index=False))

mean_acc = results_df['accuracy'].mean()
std_acc = results_df['accuracy'].std()

print(f"\n{'='*40}")
print(f"FINE-TUNED MODEL RESULTS")
print(f"{'='*40}")
print(f"Mean Accuracy: {mean_acc:.2%} ± {std_acc:.2%}")

results_df.to_csv(METRICS_FOLDER / "cv_finetuned_model_results.csv", index=False)

## Step 8: Confusion Matrix

In [ ]:
cm = confusion_matrix(all_true, all_pred, labels=[0, 1])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Child', 'Adult'],
            yticklabels=['Child', 'Adult'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'5-Fold CV - Fine-tuned Model\nAccuracy: {mean_acc:.2%} ± {std_acc:.2%}')
plt.tight_layout()
plt.savefig(METRICS_FOLDER / "cv_finetuned_confusion_matrix.png", dpi=150)
plt.show()

print(classification_report(all_true, all_pred, target_names=['Child', 'Adult']))

---
## ✅ Complete!

**Note:** This uses the SAME model for all folds (inference only, no training).
This shows how well the previously trained model generalizes to unseen videos.